# LangGraph로 금융 Agent 설계하기 

- LangGraph 자체가 AI/Agent의 실행 흐름을 관리하는 그래프 프레임워크
- 먼저 State → Node → Edge → Routing → Checkpoint → HITL → Streaming → Multi-Agent → Critic을 Python 함수만으로 확실히 이해하도록 구성했습니다.

In [ ]:
# LangGraph가 LLM을 대체하는 것이 아닙니다.
# LangGraph = Agent의 업무 흐름을 조직
# LLM = 판단·분류·해석·보고서 작성
# Python/Tool = 데이터 조회·정량 계산

LangGraph
        ┌─────────────────┐
        │ State           │
        │ Node            │
        │ Edge            │
        │ Routing         │
        │ Loop            │
        │ Checkpoint      │
        │ Human Review    │
        └────────┬────────┘
                 │
          실행 흐름 관리
                 │
       ┌─────────┴─────────┐
       ▼                   ▼
    Python/Tool            LLM
       │                   │
       │                   │
   계산/조회             판단/해석
       │                   │
       └─────────┬─────────┘
                 ▼
              Report

In [ ]:
# 준비

In [3]:
%pip install -U langgraph


   --- ------------------------------------  1/12 [uuid-utils]
   ---------- -----------------------------  3/12 [orjson]
   ------------- --------------------------  4/12 [jsonpatch]
   ---------------- -----------------------  5/12 [requests-toolbelt]
   ---------------- -----------------------  5/12 [requests-toolbelt]
   ---------------- -----------------------  5/12 [requests-toolbelt]
   ---------------- -----------------------  5/12 [requests-toolbelt]
   -------------------- -------------------  6/12 [langsmith]
   -------------------- -------------------  6/12 [langsmith]
   -------------------- -------------------  6/12 [langsmith]
   -------------------- -------------------  6/12 [langsmith]
   -------------------- -------------------  6/12 [langsmith]
   -------------------- -------------------  6/12 [langsmith]
   -------------------- -------------------  6/12 [langsmith]
   -------------------- -------------------  6/12 [langsmith]
   -------------------- ---------------

In [4]:
import langgraph

print("LangGraph import 성공")

LangGraph import 성공


## 11회차 — LangGraph / StateGraph

In [5]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class FinanceState(TypedDict):
    company: str
    request: str
    result: str


def analyze(state: FinanceState):
    return {
        "result": f"{state['company']} 분석 준비 완료"
    }


builder = StateGraph(FinanceState)

builder.add_node("analyze", analyze)

builder.add_edge(START, "analyze")
builder.add_edge("analyze", END)

graph = builder.compile()


result = graph.invoke({
    "company": "A기업",
    "request": "투자매력도"
})

print(result)

{'company': 'A기업', 'request': '투자매력도', 'result': 'A기업 분석 준비 완료'}


## 12회차 — State / Reducer

In [ ]:
# 여기서 evidence, risks는 Reducer를 이용해 덮어쓰지 않고 누적하는 구조입니다. 
#     PPT도 company / market / fundamentals / evidence / risks / report를 Financial State의 주요 항목으로 정의합니다.

In [7]:
# Reducer가 리스트를 누적하는 모습까지 확인

In [10]:
from operator import add
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END


# 1. State 정의
class FinanceState(TypedDict):
    company: str
    evidence: Annotated[list[str], add]
    risks: Annotated[list[str], add]


# 2. 첫 번째 Node
def market_analysis(state: FinanceState):
    return {
        "evidence": ["최근 1년 주가 변동성이 높음"],
        "risks": ["시장 변동성 위험"]
    }


# 3. 두 번째 Node
def financial_analysis(state: FinanceState):
    return {
        "evidence": ["영업이익률이 전년 대비 하락"],
        "risks": ["수익성 악화 가능성"]
    }


# 4. Graph 만들기
builder = StateGraph(FinanceState)

builder.add_node("market", market_analysis)
builder.add_node("financial", financial_analysis)

builder.add_edge(START, "market")
builder.add_edge("market", "financial")
builder.add_edge("financial", END)

graph = builder.compile()


# 5. 실행
result = graph.invoke({
    "company": "A기업",
    "evidence": [],
    "risks": []
})


# 6. 결과 출력
print(result)

{'company': 'A기업', 'evidence': ['최근 1년 주가 변동성이 높음', '영업이익률이 전년 대비 하락'], 'risks': ['시장 변동성 위험', '수익성 악화 가능성']}


In [ ]:
# 일반적인 State라면 뒤 Node가 값을 반환할 때 기존 값을 덮어씁니다. 
#     그런데 Annotated[..., add]를 사용하면 이전 결과에 새로운 결과가 누적됩니다. 
#     PPT에서도 Reducer를 “덮어쓰기와 누적을 구분해 병렬/반복 실행 시 충돌을 제어”하는 개념으로 설명하고 있습니다.

## 13회차 — Conditional Routing

In [ ]:
# 13회차 핵심을 state 값에 따라 market / fundamental / research 경로를 선택하는 것으로 정의하고 있습니다

In [13]:
# 즉 질문을 보고

# 주가 질문 → Market
# 재무 질문 → Fundamental
# 기타 질문 → Research

# 로 보내는 구조입니다.

In [15]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


# ========================================
# 1. State 정의
# ========================================

class FinanceState(TypedDict):
    request: str
    analysis: str
    result: str


# ========================================
# 2. Router Node
# ========================================

def router(state: FinanceState):
    print("▶ Router 실행")
    print("질문:", state["request"])

    # 여기서는 State를 변경하지 않음
    return {}


# ========================================
# 3. Conditional Routing 함수
# ========================================

def route(state: FinanceState):

    q = state["request"].lower()

    if "주가" in q:
        return "market"

    if "재무" in q:
        return "fundamental"

    return "research"


# ========================================
# 4. 각 분석 Node
# ========================================

def market_analysis(state: FinanceState):

    print("▶ Market Node 실행")

    return {
        "analysis":
            "최근 주가 흐름과 수익률을 분석했습니다."
    }


def fundamental_analysis(state: FinanceState):

    print("▶ Fundamental Node 실행")

    return {
        "analysis":
            "매출, 영업이익, 부채비율 등 재무지표를 분석했습니다."
    }


def research_analysis(state: FinanceState):

    print("▶ Research Node 실행")

    return {
        "analysis":
            "기업 관련 뉴스와 공시를 조사했습니다."
    }


# ========================================
# 5. Summarize Node
# ========================================

def summarize(state: FinanceState):

    print("▶ Summarize Node 실행")

    return {
        "result":
            f"최종 분석 결과: {state['analysis']}"
    }


# ========================================
# 6. Graph 생성
# ========================================

builder = StateGraph(FinanceState)


# Node 등록

builder.add_node(
    "router",
    router
)

builder.add_node(
    "market",
    market_analysis
)

builder.add_node(
    "fundamental",
    fundamental_analysis
)

builder.add_node(
    "research",
    research_analysis
)

builder.add_node(
    "summarize",
    summarize
)


# ========================================
# 7. Edge 연결
# ========================================

builder.add_edge(
    START,
    "router"
)


# 조건부 분기

builder.add_conditional_edges(
    "router",
    route,
    {
        "market": "market",
        "fundamental": "fundamental",
        "research": "research"
    }
)


# 각 분석 Node 이후 summarize로 합류

builder.add_edge(
    "market",
    "summarize"
)

builder.add_edge(
    "fundamental",
    "summarize"
)

builder.add_edge(
    "research",
    "summarize"
)


builder.add_edge(
    "summarize",
    END
)


# ========================================
# 8. Compile
# ========================================

graph = builder.compile()


# ========================================
# 9. 실행
# ========================================

result = graph.invoke({
    "request":
        "삼성전자 주가를 분석해줘",

    "analysis":
        "",

    "result":
        ""
})


print("\n=== 최종 State ===")

print(result)

▶ Router 실행
질문: 삼성전자 주가를 분석해줘
▶ Market Node 실행
▶ Summarize Node 실행

=== 최종 State ===
{'request': '삼성전자 주가를 분석해줘', 'analysis': '최근 주가 흐름과 수익률을 분석했습니다.', 'result': '최종 분석 결과: 최근 주가 흐름과 수익률을 분석했습니다.'}


In [16]:
result = graph.invoke({
    "request": "삼성전자 재무 상태를 분석해줘",
    "analysis": "",
    "result": ""
})

print(result)

▶ Router 실행
질문: 삼성전자 재무 상태를 분석해줘
▶ Fundamental Node 실행
▶ Summarize Node 실행
{'request': '삼성전자 재무 상태를 분석해줘', 'analysis': '매출, 영업이익, 부채비율 등 재무지표를 분석했습니다.', 'result': '최종 분석 결과: 매출, 영업이익, 부채비율 등 재무지표를 분석했습니다.'}


In [ ]:
# 이번에는 실행 경로가

# START
#   ↓
# Router
#   ↓
# "재무" 발견
#   ↓
# Fundamental
#   ↓
# Summarize
#   ↓
# END

# 가 됩니다.

In [17]:
test_questions = [
    "삼성전자 주가를 분석해줘",
    "삼성전자 재무 상태를 분석해줘",
    "삼성전자 최근 뉴스를 조사해줘"
]


for question in test_questions:

    print("\n============================")
    print("질문:", question)
    print("============================")

    result = graph.invoke({
        "request": question,
        "analysis": "",
        "result": ""
    })

    print("결과:", result["result"])


질문: 삼성전자 주가를 분석해줘
▶ Router 실행
질문: 삼성전자 주가를 분석해줘
▶ Market Node 실행
▶ Summarize Node 실행
결과: 최종 분석 결과: 최근 주가 흐름과 수익률을 분석했습니다.

질문: 삼성전자 재무 상태를 분석해줘
▶ Router 실행
질문: 삼성전자 재무 상태를 분석해줘
▶ Fundamental Node 실행
▶ Summarize Node 실행
결과: 최종 분석 결과: 매출, 영업이익, 부채비율 등 재무지표를 분석했습니다.

질문: 삼성전자 최근 뉴스를 조사해줘
▶ Router 실행
질문: 삼성전자 최근 뉴스를 조사해줘
▶ Research Node 실행
▶ Summarize Node 실행
결과: 최종 분석 결과: 기업 관련 뉴스와 공시를 조사했습니다.


## 14회차 — ToolNode로 금융 Tool 연결하기

In [ ]:
# 사용자 질문
#    ↓
# Assistant
#    ↓
# Tool 필요 판단
#    ↓
# get_stock_price
#    ↓
# ToolNode
#    ↓
# Assistant
#    ↓
# 최종 결과

In [19]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


# ==========================================
# 1. State
# ==========================================

class FinanceState(TypedDict):
    messages: Annotated[list, add_messages]


# ==========================================
# 2. 금융 Tool
# ==========================================

@tool
def get_stock_price(company: str) -> str:
    """기업의 주가 정보를 조회한다."""
    
    sample_data = {
        "삼성전자": 85000,
        "SK하이닉스": 210000,
        "현대차": 250000
    }

    price = sample_data.get(company, 100000)

    return f"{company} 현재 샘플 주가: {price:,}원"


@tool
def get_financial_ratio(company: str) -> str:
    """기업의 주요 재무비율을 조회한다."""

    return (
        f"{company} 샘플 재무비율: "
        "ROE 12.5%, 영업이익률 15.2%, 부채비율 35%"
    )


tools = [
    get_stock_price,
    get_financial_ratio
]


# ==========================================
# 3. Assistant Node
# API 없이 Tool Call 생성
# ==========================================

def assistant(state: FinanceState):

    last_message = state["messages"][-1]

    # Tool 실행 후 돌아온 경우
    if last_message.type == "tool":

        return {
            "messages": [
                AIMessage(
                    content=f"Tool 실행 결과를 확인했습니다: {last_message.content}"
                )
            ]
        }

    question = last_message.content

    # 주가 질문
    if "주가" in question:

        return {
            "messages": [
                AIMessage(
                    content="",
                    tool_calls=[
                        {
                            "name": "get_stock_price",
                            "args": {
                                "company": "삼성전자"
                            },
                            "id": "call_stock_1",
                            "type": "tool_call"
                        }
                    ]
                )
            ]
        }

    # 재무 질문
    if "재무" in question:

        return {
            "messages": [
                AIMessage(
                    content="",
                    tool_calls=[
                        {
                            "name": "get_financial_ratio",
                            "args": {
                                "company": "삼성전자"
                            },
                            "id": "call_financial_1",
                            "type": "tool_call"
                        }
                    ]
                )
            ]
        }

    return {
        "messages": [
            AIMessage(
                content="주가 또는 재무 관련 질문을 입력해주세요."
            )
        ]
    }


# ==========================================
# 4. Graph
# ==========================================

builder = StateGraph(FinanceState)

builder.add_node(
    "assistant",
    assistant
)

builder.add_node(
    "tools",
    ToolNode(tools)
)

builder.add_edge(
    START,
    "assistant"
)

builder.add_conditional_edges(
    "assistant",
    tools_condition
)

builder.add_edge(
    "tools",
    "assistant"
)

graph = builder.compile()


# ==========================================
# 5. 실행
# ==========================================

result = graph.invoke({
    "messages": [
        HumanMessage(
            content="삼성전자 주가를 분석해줘"
        )
    ]
})


print("=== 전체 메시지 ===")

for message in result["messages"]:

    print(
        message.type,
        ":",
        message.content
    )

=== 전체 메시지 ===
human : 삼성전자 주가를 분석해줘
ai : 
tool : 삼성전자 현재 샘플 주가: 85,000원
ai : Tool 실행 결과를 확인했습니다: 삼성전자 현재 샘플 주가: 85,000원


In [ ]:
# 핵심은

# Assistant
#    ↓
# Tool 필요?
#  ↓     ↓
# Yes    No
#  ↓     ↓
# Tool   END
#  ↓
# Assistant

# 라는 Tool Loop입니다. PPT에서는 Market/Fundamental Tool을 이 구조로 통합하도록 설계되어 있습니다.

## 15회차 — Persistence / Checkpoint

In [ ]:
# PPT의 15회차는 InMemorySaver, thread_id, get_state(), get_state_history()가 핵심입니다.

In [ ]:
# # 여기서 핵심은:
# thread_id = "samsung-analysis-001"입니다. 
#     공식 문서에서도 checkpointer를 사용한 graph는 thread를 기준으로 상태를 저장합니다.

In [20]:
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


# ==========================================
# 1. State
# ==========================================

class FinanceState(TypedDict):
    company: str
    step: int
    result: str


# ==========================================
# 2. Node
# ==========================================

def analyze_market(state: FinanceState):

    print("▶ Market Analysis 실행")

    return {
        "step": 1,
        "result": f"{state['company']} 시장 분석 완료"
    }


def analyze_financial(state: FinanceState):

    print("▶ Financial Analysis 실행")

    return {
        "step": 2,
        "result": (
            state["result"]
            + " → 재무 분석 완료"
        )
    }


# ==========================================
# 3. Graph
# ==========================================

builder = StateGraph(FinanceState)

builder.add_node(
    "market",
    analyze_market
)

builder.add_node(
    "financial",
    analyze_financial
)

builder.add_edge(
    START,
    "market"
)

builder.add_edge(
    "market",
    "financial"
)

builder.add_edge(
    "financial",
    END
)


# ==========================================
# 4. Checkpointer
# ==========================================

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


# ==========================================
# 5. thread_id
# ==========================================

config = {
    "configurable": {
        "thread_id": "samsung-analysis-001"
    }
}


# ==========================================
# 6. 실행
# ==========================================

result = graph.invoke(
    {
        "company": "삼성전자",
        "step": 0,
        "result": ""
    },
    config
)


print("\n=== 최종 결과 ===")
print(result)


# ==========================================
# 7. 현재 State
# ==========================================

current = graph.get_state(config)

print("\n=== 현재 저장 State ===")
print(current.values)


# ==========================================
# 8. State History
# ==========================================

print("\n=== State History ===")

for snapshot in graph.get_state_history(config):

    print(
        "다음 Node:",
        snapshot.next
    )

    print(
        "State:",
        snapshot.values
    )

    print("-" * 50)

▶ Market Analysis 실행
▶ Financial Analysis 실행

=== 최종 결과 ===
{'company': '삼성전자', 'step': 2, 'result': '삼성전자 시장 분석 완료 → 재무 분석 완료'}

=== 현재 저장 State ===
{'company': '삼성전자', 'step': 2, 'result': '삼성전자 시장 분석 완료 → 재무 분석 완료'}

=== State History ===
다음 Node: ()
State: {'company': '삼성전자', 'step': 2, 'result': '삼성전자 시장 분석 완료 → 재무 분석 완료'}
--------------------------------------------------
다음 Node: ('financial',)
State: {'company': '삼성전자', 'step': 1, 'result': '삼성전자 시장 분석 완료'}
--------------------------------------------------
다음 Node: ('market',)
State: {'company': '삼성전자', 'step': 0, 'result': ''}
--------------------------------------------------
다음 Node: ('__start__',)
State: {}
--------------------------------------------------


In [ ]:
# 여기서 핵심은 thread_id입니다.

# "thread_id": "equity-A-001"

# 같은 ID를 사용하면 특정 기업 분석의 상태를 저장하고 이어서 실행할 수 있습니다.

## 16회차 — Human-in-the-loop

In [ ]:
# 보고서 작성
#     ↓
# interrupt
#     ↓
# 사람 검토
#  ┌──┴──┐
# 승인    반려
#  ↓       ↓
# 계속    수정

# 구조입니다. PPT의 실습도 투자의견 확정 전에 애널리스트 승인 단계를 넣는 것입니다.

In [21]:
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


# ==========================================
# 1. State
# ==========================================

class FinanceState(TypedDict):
    company: str
    report: str
    approved: bool


# ==========================================
# 2. 보고서 작성 Node
# ==========================================

def write_report(state: FinanceState):

    report = (
        f"{state['company']} 분석 결과\n"
        "- 수익성: 양호\n"
        "- 재무 안정성: 양호\n"
        "- 주요 위험: 시장 변동성"
    )

    print("▶ 보고서 초안 작성")

    return {
        "report": report
    }


# ==========================================
# 3. 사람 승인 Node
# ==========================================

def approval(state: FinanceState):

    decision = interrupt({
        "company": state["company"],
        "report": state["report"],
        "question": "이 보고서를 승인하시겠습니까?"
    })

    return {
        "approved":
            decision == "approve"
    }


# ==========================================
# 4. 최종 Node
# ==========================================

def finalize(state: FinanceState):

    if state["approved"]:

        print("▶ 보고서 승인 완료")

        return {
            "report":
                state["report"]
                + "\n\n[최종 승인 완료]"
        }

    return {
        "report":
            state["report"]
            + "\n\n[승인되지 않음]"
    }


# ==========================================
# 5. Graph
# ==========================================

builder = StateGraph(FinanceState)

builder.add_node(
    "writer",
    write_report
)

builder.add_node(
    "approval",
    approval
)

builder.add_node(
    "finalize",
    finalize
)

builder.add_edge(
    START,
    "writer"
)

builder.add_edge(
    "writer",
    "approval"
)

builder.add_edge(
    "approval",
    "finalize"
)

builder.add_edge(
    "finalize",
    END
)


checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


# ==========================================
# 6. Thread
# ==========================================

config = {
    "configurable": {
        "thread_id": "approval-001"
    }
}


# ==========================================
# 7. 첫 실행
# interrupt에서 멈춤
# ==========================================

result = graph.invoke(
    {
        "company": "삼성전자",
        "report": "",
        "approved": False
    },
    config
)


print("\n=== Interrupt 결과 ===")

print(result)

▶ 보고서 초안 작성

=== Interrupt 결과 ===
{'company': '삼성전자', 'report': '삼성전자 분석 결과\n- 수익성: 양호\n- 재무 안정성: 양호\n- 주요 위험: 시장 변동성', 'approved': False, '__interrupt__': [Interrupt(value={'company': '삼성전자', 'report': '삼성전자 분석 결과\n- 수익성: 양호\n- 재무 안정성: 양호\n- 주요 위험: 시장 변동성', 'question': '이 보고서를 승인하시겠습니까?'}, id='89ec8f1378281abc6378957b85f286b6')]}


In [ ]:
# 공식 문서에서도 interrupt()로 정지하고 Command(resume=...)로 같은 thread를 이어가는 구조를 사용합니다.

## 17회차 — Streaming

In [ ]:
# PPT는 이 회차에서 Updates / Values / Messages를 구분해 실행 과정을 관찰하도록 구성합니다.

# 17회차에서는 실행 결과뿐 아니라 어느 Node가 실행되고 있는지 실시간으로 봅니다.

In [ ]:
# 개념적으로는

# market 완료
#     ↓
# fundamental 완료
#     ↓
# research 완료
#     ↓
# report 작성

# 처럼 Agent가 현재 어디까지 실행됐는지 관찰하는 회차입니다.

In [22]:
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END


class FinanceState(TypedDict):
    company: str
    market: str
    financial: str
    report: str


# ==========================================
# Nodes
# ==========================================

def market(state):

    return {
        "market":
            f"{state['company']} 주가 분석 완료"
    }


def financial(state):

    return {
        "financial":
            f"{state['company']} 재무 분석 완료"
    }


def report(state):

    return {
        "report":
            (
                state["market"]
                + " / "
                + state["financial"]
            )
    }


# ==========================================
# Graph
# ==========================================

builder = StateGraph(FinanceState)

builder.add_node(
    "market",
    market
)

builder.add_node(
    "financial",
    financial
)

builder.add_node(
    "report",
    report
)

builder.add_edge(
    START,
    "market"
)

builder.add_edge(
    "market",
    "financial"
)

builder.add_edge(
    "financial",
    "report"
)

builder.add_edge(
    "report",
    END
)

graph = builder.compile()


# ==========================================
# Streaming
# ==========================================

input_state = {
    "company": "삼성전자",
    "market": "",
    "financial": "",
    "report": ""
}


print("=== Streaming 시작 ===\n")


for chunk in graph.stream(
    input_state,
    stream_mode="updates"
):

    print(chunk)

=== Streaming 시작 ===

{'market': {'market': '삼성전자 주가 분석 완료'}}
{'financial': {'financial': '삼성전자 재무 분석 완료'}}
{'report': {'report': '삼성전자 주가 분석 완료 / 삼성전자 재무 분석 완료'}}


In [ ]:
# updates는 각 Node가 이번 단계에서 변경한 부분을 보여줍니다. 전체 누적 State를 보고 싶으면:

In [23]:
for chunk in graph.stream(
    input_state,
    stream_mode="values"
):

    print(chunk)

{'company': '삼성전자', 'market': '', 'financial': '', 'report': ''}
{'company': '삼성전자', 'market': '삼성전자 주가 분석 완료', 'financial': '', 'report': ''}
{'company': '삼성전자', 'market': '삼성전자 주가 분석 완료', 'financial': '삼성전자 재무 분석 완료', 'report': ''}
{'company': '삼성전자', 'market': '삼성전자 주가 분석 완료', 'financial': '삼성전자 재무 분석 완료', 'report': '삼성전자 주가 분석 완료 / 삼성전자 재무 분석 완료'}


In [ ]:
# 공식 문서도 updates와 values를 서로 다른 streaming mode로 구분합니다.

## 18회차 — Subgraph / Multi-Agent

In [ ]:
# PPT에서는 Market / Fundamental / Research를 각각 전문 Graph로 만들고 Manager가 합치는 구조입니다.

In [ ]:
# 즉 하나의 Agent가 모든 일을 하는 것이 아니라,

#         ┌─ Market Agent
# START ──┼─ Fundamental Agent
#         └─ Research Agent
#                ↓
#             Manager
#                ↓
#              Report

# 구조로 만듭니다. PPT에서는 이를 3개 전문 Subgraph가 협업하는 AI 투자분석팀으로 정의합니다.

In [25]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


# ==========================================
# 1. 전체 State
# ==========================================

class FinanceState(TypedDict):

    company: str

    market_result: str

    fundamental_result: str

    research_result: str

    final_report: str


# ==========================================
# 2. Market Agent
# ==========================================

def market_agent(state: FinanceState):

    print("▶ Market Agent 실행")

    company = state["company"]

    return {
        "market_result":
            f"{company} 최근 주가 수익률 +8%, "
            f"변동성 20%, MDD -15%"
    }


# ==========================================
# 3. Fundamental Agent
# ==========================================

def fundamental_agent(state: FinanceState):

    print("▶ Fundamental Agent 실행")

    company = state["company"]

    return {
        "fundamental_result":
            f"{company} ROE 12%, "
            f"영업이익률 15%, "
            f"부채비율 35%"
    }


# ==========================================
# 4. Research Agent
# ==========================================

def research_agent(state: FinanceState):

    print("▶ Research Agent 실행")

    company = state["company"]

    return {
        "research_result":
            f"{company} AI 반도체 수요 확대가 "
            f"긍정적 Catalyst로 판단됨"
    }


# ==========================================
# 5. Manager Agent
# ==========================================

def manager(state: FinanceState):

    print("▶ Manager Agent 실행")

    report = f"""
=================================
AI 투자분석 보고서
=================================

기업:
{state['company']}

[Market Analysis]
{state['market_result']}

[Fundamental Analysis]
{state['fundamental_result']}

[Research]
{state['research_result']}

[종합 의견]
시장, 재무, 뉴스 정보를 종합하여
투자 포인트와 위험요인을 함께 검토해야 합니다.
"""

    return {
        "final_report": report
    }


# ==========================================
# 6. Parent Graph
# ==========================================

builder = StateGraph(FinanceState)


builder.add_node(
    "market",
    market_agent
)

builder.add_node(
    "fundamental",
    fundamental_agent
)

builder.add_node(
    "research",
    research_agent
)

builder.add_node(
    "manager",
    manager
)


# ==========================================
# 7. 병렬 실행
# ==========================================

builder.add_edge(
    START,
    "market"
)

builder.add_edge(
    START,
    "fundamental"
)

builder.add_edge(
    START,
    "research"
)


# ==========================================
# 8. 세 분석 완료 후 Manager
# ==========================================

builder.add_edge(
    [
        "market",
        "fundamental",
        "research"
    ],
    "manager"
)


builder.add_edge(
    "manager",
    END
)


# ==========================================
# 9. Compile
# ==========================================

graph = builder.compile()


# ==========================================
# 10. 실행
# ==========================================

result = graph.invoke({

    "company":
        "삼성전자",

    "market_result":
        "",

    "fundamental_result":
        "",

    "research_result":
        "",

    "final_report":
        ""
})


# ==========================================
# 11. 결과 출력
# ==========================================

print("\n")
print(result["final_report"])

▶ Fundamental Agent 실행▶ Market Agent 실행

▶ Research Agent 실행
▶ Manager Agent 실행



AI 투자분석 보고서

기업:
삼성전자

[Market Analysis]
삼성전자 최근 주가 수익률 +8%, 변동성 20%, MDD -15%

[Fundamental Analysis]
삼성전자 ROE 12%, 영업이익률 15%, 부채비율 35%

[Research]
삼성전자 AI 반도체 수요 확대가 긍정적 Catalyst로 판단됨

[종합 의견]
시장, 재무, 뉴스 정보를 종합하여
투자 포인트와 위험요인을 함께 검토해야 합니다.



## 19회차 — Critic / Revision

In [ ]:
# 구조는:

# Writer
#   ↓
# Critic
#   ↓
# 통과? ─ Yes → Final
#   │
#   No
#   ↓
# Revision
#   ↓
# Writer

# 그리고 2회 이상 실패하면 사람에게 넘기는 구조입니다. PPT 역시 “최대 2회 자동 수정 후 사람에게 넘기기”를 실습 목표로 명시합니다.

In [26]:
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END


# ==========================================
# State
# ==========================================

class FinanceState(TypedDict):
    report: str

    critic_passed: bool

    critic_message: str

    revision_count: int


# ==========================================
# Writer
# ==========================================

def writer(state):

    revision = state["revision_count"]

    print(
        f"▶ Writer 실행 / 수정 횟수: {revision}"
    )

    # 최초 보고서
    if revision == 0:

        report = """
삼성전자는 무조건 주가가 상승할 것입니다.
데이터 출처는 생략합니다.
"""

    else:

        report = """
삼성전자는 양호한 재무성과를 보이고 있으나,
향후 주가 방향에는 불확실성이 존재합니다.

분석 기준일: 2026-08-18
출처: 샘플 금융 데이터
"""

    return {
        "report": report
    }


# ==========================================
# Critic
# ==========================================

def critic(state):

    report = state["report"]

    problems = []

    if "무조건" in report:
        problems.append(
            "과도한 투자 단정"
        )

    if "출처:" not in report:
        problems.append(
            "출처 누락"
        )

    if "기준일:" not in report:
        problems.append(
            "기준일 누락"
        )

    passed = len(problems) == 0

    print(
        "▶ Critic:",
        "통과" if passed else problems
    )

    return {
        "critic_passed":
            passed,

        "critic_message":
            ", ".join(problems)
    }


# ==========================================
# Revision Count
# ==========================================

def revision(state):

    print("▶ 수정 요청")

    return {
        "revision_count":
            state["revision_count"] + 1
    }


# ==========================================
# Human Review
# ==========================================

def human_review(state):

    print(
        "▶ 자동 수정 한도 초과 → 사람 검토"
    )

    return {}


# ==========================================
# Final
# ==========================================

def finalize(state):

    print("▶ 최종 보고서 확정")

    return {}


# ==========================================
# Router
# ==========================================

def route_after_critic(state):

    if state["critic_passed"]:
        return "finalize"

    if state["revision_count"] >= 2:
        return "human_review"

    return "revision"


# ==========================================
# Graph
# ==========================================

builder = StateGraph(FinanceState)

builder.add_node(
    "writer",
    writer
)

builder.add_node(
    "critic",
    critic
)

builder.add_node(
    "revision",
    revision
)

builder.add_node(
    "human_review",
    human_review
)

builder.add_node(
    "finalize",
    finalize
)


builder.add_edge(
    START,
    "writer"
)

builder.add_edge(
    "writer",
    "critic"
)


builder.add_conditional_edges(
    "critic",
    route_after_critic,
    {
        "finalize":
            "finalize",

        "revision":
            "revision",

        "human_review":
            "human_review"
    }
)


builder.add_edge(
    "revision",
    "writer"
)

builder.add_edge(
    "finalize",
    END
)

builder.add_edge(
    "human_review",
    END
)


graph = builder.compile()


# ==========================================
# 실행
# ==========================================

result = graph.invoke({
    "report":
        "",

    "critic_passed":
        False,

    "critic_message":
        "",

    "revision_count":
        0
})


print("\n======================")
print("최종 보고서")
print("======================")

print(
    result["report"]
)

▶ Writer 실행 / 수정 횟수: 0
▶ Critic: ['과도한 투자 단정', '출처 누락', '기준일 누락']
▶ 수정 요청
▶ Writer 실행 / 수정 횟수: 1
▶ Critic: 통과
▶ 최종 보고서 확정

최종 보고서

삼성전자는 양호한 재무성과를 보이고 있으나,
향후 주가 방향에는 불확실성이 존재합니다.

분석 기준일: 2026-08-18
출처: 샘플 금융 데이터



## 20회차 — 최종 프로젝트: Equity Research Graph

In [ ]:
# 20회차는 앞의 모든 요소를 합치는 구조입니다. PPT의 과정 설계상 최종 산출물은 Equity Research Graph입니다.

In [ ]:
# 최종적으로 수업의 전체 Python 구조를 한 줄로 정리하면:

# 11 StateGraph
#       ↓
# 12 Financial State
#       ↓
# 13 Conditional Routing
#       ↓
# 14 ToolNode
#       ↓
# 15 Checkpoint
#       ↓
# 16 Human Approval
#       ↓
# 17 Streaming
#       ↓
# 18 Multi-Agent
#       ↓
# 19 Critic / Revision
#       ↓
# 20 Equity Research Agent

In [29]:
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


# ==========================================
# 1. 전체 State
# ==========================================

class EquityResearchState(TypedDict):
    company: str
    market: str
    fundamental: str
    research: str
    draft_report: str
    critic_passed: bool
    critic_message: str
    revision_count: int
    final_report: str


# ==========================================
# 2. Market Agent
# ==========================================

def market_agent(state: EquityResearchState):

    print("▶ Market Agent")

    return {
        "market":
            f"{state['company']} 최근 1년 수익률 +8%, "
            "변동성 20%, MDD -15%"
    }


# ==========================================
# 3. Fundamental Agent
# ==========================================

def fundamental_agent(state: EquityResearchState):

    print("▶ Fundamental Agent")

    return {
        "fundamental":
            f"{state['company']} ROE 12.5%, "
            "영업이익률 15%, 부채비율 35%"
    }


# ==========================================
# 4. Research Agent
# ==========================================

def research_agent(state: EquityResearchState):

    print("▶ Research Agent")

    return {
        "research":
            f"{state['company']} AI 관련 수요 확대가 "
            "긍정적 Catalyst"
    }


# ==========================================
# 5. Report Writer
# ==========================================

def writer(state: EquityResearchState):

    print("▶ Report Writer")

    report = f"""
================================
AI Equity Research Report
================================

기업
{state['company']}

[1. Market]
{state['market']}

[2. Fundamental]
{state['fundamental']}

[3. News & Catalyst]
{state['research']}

[4. Investment Thesis]
재무성과와 성장 Catalyst는 긍정적입니다.
다만 시장 변동성과 산업 위험을 함께 고려해야 합니다.

분석 기준일: 2026-08-18
출처: 교육용 샘플 데이터
"""

    return {
        "draft_report": report
    }


# ==========================================
# 6. Critic
# ==========================================

def critic(state: EquityResearchState):

    print("▶ Critic")

    report = state["draft_report"]

    problems = []

    if "분석 기준일:" not in report:
        problems.append("기준일 누락")

    if "출처:" not in report:
        problems.append("출처 누락")

    if "무조건" in report:
        problems.append("과도한 투자 단정")

    passed = len(problems) == 0

    print(
        "검증 결과:",
        "PASS" if passed else problems
    )

    return {
        "critic_passed": passed,
        "critic_message": ", ".join(problems)
    }


# ==========================================
# 7. Revision
# ==========================================

def revise(state: EquityResearchState):

    print("▶ Report Revision")

    revised = (
        state["draft_report"]
        + "\n[검증 결과를 반영하여 수정 완료]"
    )

    return {
        "draft_report": revised,
        "revision_count":
            state["revision_count"] + 1
    }


# ==========================================
# 8. Finalize
# ==========================================

def finalize(state: EquityResearchState):

    print("▶ Finalize")

    return {
        "final_report":
            state["draft_report"]
    }


# ==========================================
# 9. Critic Router
# ==========================================

def critic_router(state: EquityResearchState):

    if state["critic_passed"]:
        return "finalize"

    if state["revision_count"] >= 2:
        return "finalize"

    return "revise"


# ==========================================
# 10. Graph 생성
# ==========================================

builder = StateGraph(EquityResearchState)

builder.add_node(
    "market",
    market_agent
)

builder.add_node(
    "fundamental",
    fundamental_agent
)

builder.add_node(
    "research",
    research_agent
)

builder.add_node(
    "writer",
    writer
)

builder.add_node(
    "critic",
    critic
)

builder.add_node(
    "revise",
    revise
)

builder.add_node(
    "finalize",
    finalize
)


# ==========================================
# 11. 병렬 분석
# ==========================================

builder.add_edge(
    START,
    "market"
)

builder.add_edge(
    START,
    "fundamental"
)

builder.add_edge(
    START,
    "research"
)


# ==========================================
# 12. 세 Agent 완료 후 Writer
# ==========================================

builder.add_edge(
    [
        "market",
        "fundamental",
        "research"
    ],
    "writer"
)


builder.add_edge(
    "writer",
    "critic"
)


# ==========================================
# 13. Critic Loop
# ==========================================

builder.add_conditional_edges(
    "critic",
    critic_router,
    {
        "finalize": "finalize",
        "revise": "revise"
    }
)


builder.add_edge(
    "revise",
    "critic"
)

builder.add_edge(
    "finalize",
    END
)


# ==========================================
# 14. Persistence
# ==========================================

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


# ==========================================
# 15. 실행 설정
# ==========================================

config = {
    "configurable": {
        "thread_id":
            "equity-samsung-001"
    }
}


input_state = {
    "company":
        "삼성전자",

    "market":
        "",

    "fundamental":
        "",

    "research":
        "",

    "draft_report":
        "",

    "critic_passed":
        False,

    "critic_message":
        "",

    "revision_count":
        0,

    "final_report":
        ""
}


# ==========================================
# 16. Streaming 실행
# ==========================================

print(
    "=== Agent 실행 시작 ===\n"
)


for chunk in graph.stream(
    input_state,
    config,
    stream_mode="updates"
):

    print(chunk)


# ==========================================
# 17. 최종 State 조회
# ==========================================

state = graph.get_state(
    config
)


print(
    "\n=============================="
)

print(
    "최종 AI Equity Research Report"
)

print(
    "=============================="
)


print(
    state.values["final_report"]
)

=== Agent 실행 시작 ===

▶ Fundamental Agent
▶ Market Agent
▶ Research Agent
{'fundamental': {'fundamental': '삼성전자 ROE 12.5%, 영업이익률 15%, 부채비율 35%'}}
{'market': {'market': '삼성전자 최근 1년 수익률 +8%, 변동성 20%, MDD -15%'}}
{'research': {'research': '삼성전자 AI 관련 수요 확대가 긍정적 Catalyst'}}
▶ Report Writer
{'writer': {'draft_report': '\n================================\nAI Equity Research Report\n================================\n\n기업\n삼성전자\n\n[1. Market]\n삼성전자 최근 1년 수익률 +8%, 변동성 20%, MDD -15%\n\n[2. Fundamental]\n삼성전자 ROE 12.5%, 영업이익률 15%, 부채비율 35%\n\n[3. News & Catalyst]\n삼성전자 AI 관련 수요 확대가 긍정적 Catalyst\n\n[4. Investment Thesis]\n재무성과와 성장 Catalyst는 긍정적입니다.\n다만 시장 변동성과 산업 위험을 함께 고려해야 합니다.\n\n분석 기준일: 2026-08-18\n출처: 교육용 샘플 데이터\n'}}
▶ Critic
검증 결과: PASS
{'critic': {'critic_passed': True, 'critic_message': ''}}
▶ Finalize
{'finalize': {'final_report': '\n================================\nAI Equity Research Report\n================================\n\n기업\n삼성전자\n\n[1. Market]\n삼성전자 최근 1년 수익률 +8%, 변동성 20%, MDD -15

# OpenAI LLM을 실제로 사용(13회차 → 20회차 코드를 ) 하는 완전 실행형 버전

- 공통 준비 — 최초 1회

In [33]:
# %pip install -U langgraph langchain langchain-openai pydantic typing_extensions

- 커널을 재시작한 다음 API Key를 설정합니다.

In [ ]:
# import os
# import getpass

# if not os.environ.get("OPENAI_API_KEY"):
#     os.environ["OPENAI_API_KEY"] = getpass.getpass(
#         "OPENAI API Key: "
#     )

In [35]:
# from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(
#     model="gpt-5.4-mini"
# )

In [ ]:
# 현재 LangChain 공식 문서에서도 ChatOpenAI(model="gpt-5.4-mini") 형태를 사용하고 있습니다.

# 계정에서 해당 모델을 사용할 수 없다면 본인 API 계정에서 사용 가능한 모델명으로 model=만 변경하면 됩니다.

## 13회차 — LLM Conditional Routing

In [ ]:
# 기존에는 이것을 Python 규칙으로 했습니다.

# if "주가" in question:
#     return "market"

# 이제는 LLM이 질문의 의미를 판단해 Routing하게 만듭니다.

In [ ]:
# from typing import Literal
# from typing_extensions import TypedDict
# from pydantic import BaseModel

# from langchain_openai import ChatOpenAI
# from langgraph.graph import StateGraph, START, END


# # ==========================================
# # 1. LLM
# # ==========================================

# llm = ChatOpenAI(
#     model="gpt-5.4-mini"
# )


# # ==========================================
# # 2. Routing 결과 Schema
# # ==========================================

# class RouteDecision(BaseModel):
#     route: Literal[
#         "market",
#         "fundamental",
#         "research"
#     ]


# router_llm = llm.with_structured_output(
#     RouteDecision
# )


# # ==========================================
# # 3. State
# # ==========================================

# class FinanceState(TypedDict):
#     request: str
#     route: str
#     analysis: str
#     result: str


# # ==========================================
# # 4. LLM Router Node
# # ==========================================

# def router(state: FinanceState):

#     print("▶ LLM Router 실행")

# #     decision = router_llm.invoke(
#         f"""
# 다음 금융 질문을 분류하세요.

# 질문:
# {state['request']}

# 분류 기준:

# market
# - 주가
# - 수익률
# - 변동성
# - 시장성과
# - 차트

# fundamental
# - 실적
# - 재무제표
# - 매출
# - 영업이익
# - ROE
# - 부채

# research
# - 뉴스
# - 공시
# - 산업
# - 이벤트
# - 경영 이슈

# 가장 적절한 하나를 선택하세요.
# """
#     )

#     print("선택 경로:", decision.route)

#     return {
#         "route": decision.route
#     }


# # ==========================================
# # 5. Routing 함수
# # ==========================================

# def route_after_llm(state: FinanceState):

#     return state["route"]


# # ==========================================
# # 6. 각 분석 Node
# # ==========================================

# def market(state):

#     return {
#         "analysis":
#             "주가·수익률·변동성을 분석하는 "
#             "Market Agent가 선택되었습니다."
#     }


# def fundamental(state):

#     return {
#         "analysis":
#             "매출·이익·재무비율을 분석하는 "
#             "Fundamental Agent가 선택되었습니다."
#     }


# def research(state):

#     return {
#         "analysis":
#             "뉴스·공시·산업이슈를 조사하는 "
#             "Research Agent가 선택되었습니다."
#     }


# # ==========================================
# # 7. Summary
# # ==========================================

# def summarize(state):

#     return {
#         "result":
#             f"""
# 질문:
# {state['request']}

# LLM이 선택한 경로:
# {state['route']}

# 처리:
# {state['analysis']}
# """
#     }


# # ==========================================
# # 8. Graph
# # ==========================================

# builder = StateGraph(FinanceState)

# builder.add_node("router", router)
# builder.add_node("market", market)
# builder.add_node("fundamental", fundamental)
# builder.add_node("research", research)
# builder.add_node("summarize", summarize)

# builder.add_edge(
#     START,
#     "router"
# )

# builder.add_conditional_edges(
#     "router",
#     route_after_llm,
#     {
#         "market": "market",
#         "fundamental": "fundamental",
#         "research": "research"
#     }
# )

# builder.add_edge(
#     "market",
#     "summarize"
# )

# builder.add_edge(
#     "fundamental",
#     "summarize"
# )

# builder.add_edge(
#     "research",
#     "summarize"
# )

# builder.add_edge(
#     "summarize",
#     END
# )

# graph = builder.compile()


# # ==========================================
# # 9. 실행
# # ==========================================

# result = graph.invoke({
#     "request":
#         "삼성전자 실적이 앞으로 괜찮을까?",

#     "route": "",
#     "analysis": "",
#     "result": ""
# })

# print(result["result"])

## 14회차 — LLM Tool Calling + ToolNode

In [ ]:
# LangChain의 ChatOpenAI는 bind_tools()를 지원합니다.

In [ ]:
# from langchain_openai import ChatOpenAI

# from langchain_core.tools import tool
# from langchain_core.messages import SystemMessage

# from langgraph.graph import (
#     MessagesState,
#     StateGraph,
#     START
# )

# from langgraph.prebuilt import (
#     ToolNode,
#     tools_condition
# )


# # ==========================================
# # 1. 금융 Tool
# # ==========================================

# @tool
# def get_stock_price(company: str) -> str:
#     """
#     기업의 주가 정보를 조회한다.
#     """

#     sample = {
#         "삼성전자":
#             "현재가 85,000원, "
#             "1년 수익률 +8%, "
#             "변동성 20%",

#         "SK하이닉스":
#             "현재가 210,000원, "
#             "1년 수익률 +15%, "
#             "변동성 28%"
#     }

#     return sample.get(
#         company,
#         f"{company} 주가 데이터 없음"
#     )


# @tool
# def get_financial_ratio(company: str) -> str:
#     """
#     기업의 주요 재무비율을 조회한다.
#     """

#     sample = {
#         "삼성전자":
#             "ROE 12.5%, "
#             "영업이익률 15%, "
#             "부채비율 35%",

#         "SK하이닉스":
#             "ROE 10.2%, "
#             "영업이익률 18%, "
#             "부채비율 42%"
#     }

#     return sample.get(
#         company,
#         f"{company} 재무 데이터 없음"
#     )


# tools = [
#     get_stock_price,
#     get_financial_ratio
# ]


# # ==========================================
# # 2. LLM + Tool
# # ==========================================

# llm = ChatOpenAI(
#     model="gpt-5.4-mini"
# )

# llm_with_tools = llm.bind_tools(
#     tools
# )


# # ==========================================
# # 3. Assistant Node
# # ==========================================

# def assistant(state: MessagesState):

#     system = SystemMessage(
#         content="""
# 당신은 금융분석 AI Agent입니다.

# 규칙:
# 1. 숫자가 필요하면 반드시 Tool을 사용하세요.
# 2. Tool 결과에 없는 숫자를 만들지 마세요.
# 3. 계산 결과와 해석을 구분하세요.
# 4. 투자 판단을 과도하게 단정하지 마세요.
# """
#     )

#     response = llm_with_tools.invoke(
#         [
#             system,
#             *state["messages"]
#         ]
#     )

#     return {
#         "messages": [response]
#     }


# ==========================================
# 4. Graph
# ==========================================

builder = StateGraph(
    MessagesState
)

builder.add_node(
    "assistant",
    assistant
)

builder.add_node(
    "tools",
    ToolNode(tools)
)

builder.add_edge(
    START,
    "assistant"
)

builder.add_conditional_edges(
    "assistant",
    tools_condition
)

builder.add_edge(
    "tools",
    "assistant"
)

graph = builder.compile()


# ==========================================
# 5. 실행
# ==========================================

result = graph.invoke({
    "messages": [
        {
            "role": "user",
            "content":
                "삼성전자 주가와 재무상태를 함께 분석해줘"
        }
    ]
})


print("\n=== 실행 결과 ===")

for msg in result["messages"]:

    print(
        msg.type,
        ":",
        msg.content
    )

In [ ]:
# 이게 실제 Tool-using Financial Agent입니다.

## 15회차 — LLM + Persistence / Checkpoint

In [ ]:
# from langchain_openai import ChatOpenAI

# from langgraph.graph import (
#     MessagesState,
#     StateGraph,
#     START,
#     END
# )

# from langgraph.checkpoint.memory import (
#     InMemorySaver
# )


# # ==========================================
# # 1. LLM
# # ==========================================

# llm = ChatOpenAI(
#     model="gpt-5.4-mini"
# )


# # ==========================================
# # 2. Node
# # ==========================================

# def analyst(state: MessagesState):

#     response = llm.invoke(
#         state["messages"]
#     )

#     return {
#         "messages": [response]
#     }


# # ==========================================
# # 3. Graph
# # ==========================================

# builder = StateGraph(
#     MessagesState
# )

# builder.add_node(
#     "analyst",
#     analyst
# )

# builder.add_edge(
#     START,
#     "analyst"
# )

# builder.add_edge(
#     "analyst",
#     END
# )


# # ==========================================
# # 4. Checkpoint
# # ==========================================

# checkpointer = InMemorySaver()

# graph = builder.compile(
#     checkpointer=checkpointer
# )


# config = {
#     "configurable": {
#         "thread_id":
#             "finance-samsung-001"
#     }
# }


# # ==========================================
# # 5. 첫 번째 질문
# # ==========================================

# result1 = graph.invoke(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content":
#                     """
# 분석 대상 기업은 삼성전자입니다.
# 현재 상황을 간단히 분석하는
# 금융 애널리스트 역할을 해주세요.
# """
#             }
#         ]
#     },
#     config
# )


# print(
#     "첫 답변:\n",
#     result1["messages"][-1].content
# )


# # ==========================================
# # 6. 두 번째 질문
# # 같은 thread_id
# # ==========================================

# result2 = graph.invoke(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content":
#                     """
# 방금 분석한 기업의
# 핵심 리스크 3개를 정리해줘.
# 기업명도 다시 써줘.
# """
#             }
#         ]
#     },
#     config
# )


# print(
#     "\n두 번째 답변:\n",
#     result2["messages"][-1].content
# )

In [ ]:
# LangGraph persistence는 thread_id를 통해 checkpoint를 저장·조회하는 구조입니다.

# 현재 상태도 볼 수 있습니다.

# state = graph.get_state(
#     config
# )


# print(state.values)

## 16회차 — LLM 보고서 + Human-in-the-loop

In [ ]:
# 이번에는 LLM이 초안을 쓰고 사람이 승인해야 최종 확정합니다.

In [ ]:
# from typing_extensions import TypedDict

# from langchain_openai import ChatOpenAI

# from langgraph.graph import (
#     StateGraph,
#     START,
#     END
# )

# from langgraph.types import (
#     interrupt,
#     Command
# )

# from langgraph.checkpoint.memory import (
#     InMemorySaver
# )


# # ==========================================
# # 1. State
# # ==========================================

# class ReportState(TypedDict):
#     company: str
#     report: str
#     approved: bool
#     final_report: str


# # ==========================================
# # 2. LLM
# # ==========================================

# llm = ChatOpenAI(
#     model="gpt-5.4-mini"
# )


# # ==========================================
# # 3. Writer LLM
# # ==========================================

# def writer(state: ReportState):

#     response = llm.invoke(
#         f"""
# 당신은 금융 애널리스트입니다.

# 기업:
# {state['company']}

# 교육용 샘플이라고 명확히 표시하고
# 다음 구조의 간단한 보고서를 작성하세요.

# 1. Executive Summary
# 2. 투자 포인트
# 3. Risk
# 4. 결론

# 실제 데이터가 제공되지 않았으므로
# 구체적 수치를 임의 생성하지 마세요.
# """
#     )

#     return {
#         "report":
#             response.content
#     }


# # ==========================================
# # 4. Human Approval
# # ==========================================

# def approval(state: ReportState):

#     decision = interrupt({
#         "company":
#             state["company"],

#         "report":
#             state["report"],

#         "question":
#             "이 보고서를 최종 승인하시겠습니까?"
#     })

#     return {
#         "approved":
#             bool(decision)
#     }


# # ==========================================
# # 5. Finalize
# # ==========================================

# def finalize(state: ReportState):

#     if state["approved"]:

#         final = (
#             state["report"]
#             + "\n\n[Human Approved]"
#         )

#     else:

#         final = (
#             state["report"]
#             + "\n\n[Human Rejected]"
#         )

#     return {
#         "final_report": final
#     }


# # ==========================================
# # 6. Graph
# # ==========================================

# builder = StateGraph(
#     ReportState
# )

# builder.add_node(
#     "writer",
#     writer
# )

# builder.add_node(
#     "approval",
#     approval
# )

# builder.add_node(
#     "finalize",
#     finalize
# )

# builder.add_edge(
#     START,
#     "writer"
# )

# builder.add_edge(
#     "writer",
#     "approval"
# )

# builder.add_edge(
#     "approval",
#     "finalize"
# )

# builder.add_edge(
#     "finalize",
#     END
# )


# checkpointer = InMemorySaver()

# graph = builder.compile(
#     checkpointer=checkpointer
# )


# config = {
#     "configurable": {
#         "thread_id":
#             "approval-001"
#     }
# }


# # ==========================================
# # 7. 첫 실행
# # ==========================================

# result = graph.invoke(
#     {
#         "company":
#             "삼성전자",

#         "report":
#             "",

#         "approved":
#             False,

#         "final_report":
#             ""
#     },
#     config
# )


# print(result)

In [ ]:
# 여기에서 interrupt()로 멈춥니다.

# 승인하려면 다음 셀을 실행합니다.

In [ ]:
# result = graph.invoke(
#     Command(
#         resume=True
#     ),
#     config
# )

# print(
#     result["final_report"]
# )

In [ ]:
#반려:

In [ ]:
# result = graph.invoke(
#     Command(
#         resume=False
#     ),
#     config
# )

In [ ]:
# 공식 문서상 interrupt를 재개할 때는 **같은 thread_id**를 사용하고 Command(resume=...) 값을 전달합니다.

## 17회차 — LLM Token Streaming

In [ ]:
# 이번에는 LLM이 보고서를 생성하는 과정을 실시간으로 보여줍니다.

In [ ]:
# from langchain_openai import ChatOpenAI

# from langgraph.graph import (
#     MessagesState,
#     StateGraph,
#     START,
#     END
# )


# # ==========================================
# # 1. LLM
# # ==========================================

# llm = ChatOpenAI(
#     model="gpt-5.4-mini"
# )


# # ==========================================
# # 2. Node
# # ==========================================

# def analyst(state: MessagesState):

#     response = llm.invoke(
#         state["messages"]
#     )

#     return {
#         "messages": [response]
#     }


# # ==========================================
# # 3. Graph
# # ==========================================

# builder = StateGraph(
#     MessagesState
# )

# builder.add_node(
#     "analyst",
#     analyst
# )

# builder.add_edge(
#     START,
#     "analyst"
# )

# builder.add_edge(
#     "analyst",
#     END
# )

# graph = builder.compile()


# # ==========================================
# # 4. Streaming
# # ==========================================

# inputs = {
#     "messages": [
#         {
#             "role": "user",
#             "content":
#                 """
# 삼성전자를 예제로
# 금융분석 AI Agent가
# 어떤 분석 절차를 수행해야 하는지
# 간단한 보고서 형식으로 설명해줘.
# """
#         }
#     ]
# }


# for chunk in graph.stream(
#     inputs,
#     stream_mode="messages",
#     version="v2"
# ):

#     if chunk["type"] == "messages":

#         message, metadata = chunk["data"]

#         if message.content:

#             print(
#                 message.content,
#                 end="",
#                 flush=True
#             )

In [ ]:
# LangGraph의 현재 streaming API는 stream_mode="messages"를 이용해 LLM 출력 token을 스트리밍할 수 있고, updates, values 등도 지원합니다.

# Node별 변화가 궁금하면:

# for chunk in graph.stream(
#     inputs,
#     stream_mode="updates",
#     version="v2"
# ):
#     print(chunk)

## 18회차 — LLM Multi-Agent

In [ ]:
# 이제 세 개의 전문 LLM Agent가 병렬 분석하고 Manager LLM이 종합합니다.

# 이전 에러를 방지하기 위해 각 병렬 Node는 자기 key 하나만 반환합니다.

In [ ]:
# from typing_extensions import TypedDict

# from langchain_openai import ChatOpenAI

# from langgraph.graph import (
#     StateGraph,
#     START,
#     END
# )


# # ==========================================
# # 1. State
# # ==========================================

# class MultiAgentState(TypedDict):
#     company: str

#     market_result: str
#     fundamental_result: str
#     research_result: str

#     final_report: str


# # ==========================================
# # 2. LLM
# # ==========================================

# llm = ChatOpenAI(
#     model="gpt-5.4-mini"
# )


# # ==========================================
# # 3. Market Agent
# # ==========================================

# def market_agent(state):

#     print("▶ Market LLM")

#     response = llm.invoke(
#         f"""
# 당신은 Market Analyst입니다.

# 기업:
# {state['company']}

# 실제 시장 데이터가 제공되지 않았으므로
# 숫자를 임의 생성하지 마세요.

# 다음 관점에서
# 분석 시 확인해야 할 내용을 작성하세요.

# - 주가 흐름
# - 수익률
# - 변동성
# - MDD
# - Benchmark 비교
# """
#     )

#     return {
#         "market_result":
#             response.content
#     }


# # ==========================================
# # 4. Fundamental Agent
# # ==========================================

# def fundamental_agent(state):

#     print("▶ Fundamental LLM")

#     response = llm.invoke(
#         f"""
# 당신은 Fundamental Analyst입니다.

# 기업:
# {state['company']}

# 실제 재무 데이터가 제공되지 않았으므로
# 수치를 임의 생성하지 마세요.

# 다음 분석 항목을 설명하세요.

# - 성장성
# - 수익성
# - 안정성
# - 현금흐름
# - Valuation
# """
#     )

#     return {
#         "fundamental_result":
#             response.content
#     }


# # ==========================================
# # 5. Research Agent
# # ==========================================

# def research_agent(state):

#     print("▶ Research LLM")

#     response = llm.invoke(
#         f"""
# 당신은 Financial Research Analyst입니다.

# 기업:
# {state['company']}

# 현재 뉴스 검색 Tool은 연결하지 않았습니다.

# 따라서 실제 최신 뉴스를 만들지 말고
# 조사해야 할 항목을 정리하세요.

# - 공시
# - 산업 뉴스
# - Catalyst
# - Risk
# - 출처와 기준일
# """
#     )

#     return {
#         "research_result":
#             response.content
#     }


# # ==========================================
# # 6. Manager LLM
# # ==========================================

# def manager(state):

#     print("▶ Manager LLM")

#     response = llm.invoke(
#         f"""
# 당신은 Portfolio Manager입니다.

# 아래 세 전문 Analyst의 결과를
# 하나의 보고서로 통합하세요.

# 기업:
# {state['company']}

# [Market]
# {state['market_result']}

# [Fundamental]
# {state['fundamental_result']}

# [Research]
# {state['research_result']}

# 다음 형식으로 작성하세요.

# 1. Executive Summary
# 2. Market View
# 3. Fundamental View
# 4. Research View
# 5. 주요 Risk
# 6. 추가로 필요한 데이터
# """
#     )

#     return {
#         "final_report":
#             response.content
#     }


# # ==========================================
# # 7. Graph
# # ==========================================

# builder = StateGraph(
#     MultiAgentState
# )

# builder.add_node(
#     "market",
#     market_agent
# )

# builder.add_node(
#     "fundamental",
#     fundamental_agent
# )

# builder.add_node(
#     "research",
#     research_agent
# )

# builder.add_node(
#     "manager",
#     manager
# )


# # 병렬 실행

# builder.add_edge(
#     START,
#     "market"
# )

# builder.add_edge(
#     START,
#     "fundamental"
# )

# builder.add_edge(
#     START,
#     "research"
# )


# # 세 Agent 완료 후 Manager

# builder.add_edge(
#     [
#         "market",
#         "fundamental",
#         "research"
#     ],
#     "manager"
# )

# builder.add_edge(
#     "manager",
#     END
# )

# graph = builder.compile()


# # ==========================================
# # 8. 실행
# # ==========================================

# result = graph.invoke({
#     "company":
#         "삼성전자",

#     "market_result":
#         "",

#     "fundamental_result":
#         "",

#     "research_result":
#         "",

#     "final_report":
#         ""
# })


# print(
#     "\n======================"
# )

# print(
#     result["final_report"]
# )

In [ ]:
# 공식 문서상 같은 super-step에서 여러 Node가 병렬 실행될 수 있습니다.

# 구조는:

#                Market LLM
#                   │
# START ───── Fundamental LLM
#                   │
#                Research LLM
#                   │
#                   ▼
#               Manager LLM
#                   │
#                   ▼
#                 END

# 입니다.

## 19회차 — Writer LLM + Critic LLM + Revision Loop

- 이번에는 LLM이 자기 자신이 작성한 보고서를 다른 역할의 LLM으로 검토합니다.

In [ ]:
# from typing_extensions import TypedDict
# from pydantic import BaseModel

# from langchain_openai import ChatOpenAI

# from langgraph.graph import (
#     StateGraph,
#     START,
#     END
# )


# # ==========================================
# # 1. Critic Schema
# # ==========================================

# class CriticResult(BaseModel):
#     passed: bool
#     feedback: str


# # ==========================================
# # 2. State
# # ==========================================

# class CriticState(TypedDict):
#     company: str

#     report: str

#     critic_passed: bool
#     feedback: str

#     revision_count: int

#     final_report: str


# # ==========================================
# # 3. LLM
# # ==========================================

# llm = ChatOpenAI(
#     model="gpt-5.4-mini"
# )

# critic_llm = llm.with_structured_output(
#     CriticResult
# )


# # ==========================================
# # 4. Writer LLM
# # ==========================================

# def writer(state):

#     print(
#         "▶ Writer LLM / Revision:",
#         state["revision_count"]
#     )

#     prompt = f"""
# 당신은 금융 애널리스트입니다.

# 기업:
# {state['company']}

# 다음 기준으로
# 교육용 금융분석 보고서를 작성하세요.

# - 존재하지 않는 숫자를 만들지 않는다.
# - 실제 최신 데이터를 아는 척하지 않는다.
# - 필요한 데이터가 없으면 명시한다.
# - 과도한 매수/매도 단정을 피한다.
# - Risk를 반드시 포함한다.

# Critic의 이전 Feedback:
# {state['feedback']}

# 보고서를 작성하거나
# Feedback에 맞춰 수정하세요.
# """

#     response = llm.invoke(
#         prompt
#     )

#     return {
#         "report":
#             response.content
#     }


# # ==========================================
# # 5. Critic LLM
# # ==========================================

# def critic(state):

#     print("▶ Critic LLM")

#     result = critic_llm.invoke(
#         f"""
# 당신은 금융분석 보고서 검증자입니다.

# 아래 보고서를 검토하세요.

# {state['report']}

# 다음을 확인하세요.

# 1. 근거 없는 숫자가 있는가?
# 2. 최신 데이터를 아는 척하는가?
# 3. 과도한 투자 단정이 있는가?
# 4. Risk가 포함되어 있는가?
# 5. 데이터 부족 여부를 명시하는가?

# 문제가 심각하지 않으면 passed=true.
# 수정이 필요하면 passed=false와
# 구체적인 feedback을 반환하세요.
# """
#     )

#     print(
#         "PASS:",
#         result.passed
#     )

#     print(
#         "Feedback:",
#         result.feedback
#     )

#     return {
#         "critic_passed":
#             result.passed,

#         "feedback":
#             result.feedback
#     }


# # ==========================================
# # 6. Revision Count
# # ==========================================

# def prepare_revision(state):

#     return {
#         "revision_count":
#             state["revision_count"] + 1
#     }


# # ==========================================
# # 7. Final
# # ==========================================

# def finalize(state):

#     return {
#         "final_report":
#             state["report"]
#     }


# # ==========================================
# # 8. Routing
# # ==========================================

# def route_after_critic(state):

#     if state["critic_passed"]:
#         return "finalize"

#     if state["revision_count"] >= 2:
#         return "finalize"

#     return "revise"


# # ==========================================
# # 9. Graph
# # ==========================================

# builder = StateGraph(
#     CriticState
# )

# builder.add_node(
#     "writer",
#     writer
# )

# builder.add_node(
#     "critic",
#     critic
# )

# builder.add_node(
#     "revise",
#     prepare_revision
# )

# builder.add_node(
#     "finalize",
#     finalize
# )


# builder.add_edge(
#     START,
#     "writer"
# )

# builder.add_edge(
#     "writer",
#     "critic"
# )

# builder.add_conditional_edges(
#     "critic",
#     route_after_critic,
#     {
#         "finalize":
#             "finalize",

#         "revise":
#             "revise"
#     }
# )

# builder.add_edge(
#     "revise",
#     "writer"
# )

# builder.add_edge(
#     "finalize",
#     END
# )

# graph = builder.compile()


# # ==========================================
# # 10. 실행
# # ==========================================

# result = graph.invoke({
#     "company":
#         "삼성전자",

#     "report":
#         "",

#     "critic_passed":
#         False,

#     "feedback":
#         "",

#     "revision_count":
#         0,

#     "final_report":
#         ""
# })


# print(
#     "\n======================"
# )

# print(
#     "최종 보고서"
# )

# print(
#     "======================"
# )

# print(
#     result["final_report"]
# )

In [ ]:
# 구조가 이제 진짜 Agent Loop가 됩니다.

# Writer LLM
#     ↓
# Critic LLM
#     ↓
#  PASS?
#   /  \
# Yes   No
#  ↓     ↓
# Final Revision
#        ↓
#     Writer LLM

## 20회차 — LLM 기반 최종 Equity Research Agent

- 마지막은 지금까지 배운 내용을 통합합니다.
- Multi-Agent + Writer LLM + Critic LLM + Revision + Checkpoint + Human Approval입니다.

In [ ]:
from typing_extensions import TypedDict
from pydantic import BaseModel

from langchain_openai import ChatOpenAI

from langgraph.graph import (
    StateGraph,
    START,
    END
)

from langgraph.types import (
    interrupt,
    Command
)

from langgraph.checkpoint.memory import (
    InMemorySaver
)


# ==========================================
# 1. Critic 구조
# ==========================================

class CriticResult(BaseModel):
    passed: bool
    feedback: str


# ==========================================
# 2. 전체 State
# ==========================================

class EquityResearchState(TypedDict):

    company: str
    request: str

    market: str
    fundamental: str
    research: str

    draft_report: str

    critic_passed: bool
    critic_feedback: str

    revision_count: int

    approved: bool

    final_report: str


# ==========================================
# 3. LLM
# ==========================================

llm = ChatOpenAI(
    model="gpt-5.4-mini"
)

critic_llm = llm.with_structured_output(
    CriticResult
)


# ==========================================
# 4. Market Agent
# ==========================================

def market_agent(state):

    print("▶ Market Agent")

    response = llm.invoke(
        f"""
당신은 Market Analyst입니다.

기업:
{state['company']}

사용자 요청:
{state['request']}

실제 주가 데이터 Tool은
이번 예제에서는 연결되어 있지 않습니다.

따라서 숫자를 만들지 말고,
시장 분석에서 확인해야 할
핵심 항목과 해석 방향을 작성하세요.
"""
    )

    return {
        "market":
            response.content
    }


# ==========================================
# 5. Fundamental Agent
# ==========================================

def fundamental_agent(state):

    print("▶ Fundamental Agent")

    response = llm.invoke(
        f"""
당신은 Fundamental Analyst입니다.

기업:
{state['company']}

실제 재무제표 데이터가 없으므로
숫자를 임의 생성하지 마세요.

성장성, 수익성, 안정성,
현금흐름, Valuation 관점에서
확인해야 할 분석을 작성하세요.
"""
    )

    return {
        "fundamental":
            response.content
    }


# ==========================================
# 6. Research Agent
# ==========================================

def research_agent(state):

    print("▶ Research Agent")

    response = llm.invoke(
        f"""
당신은 Financial Research Analyst입니다.

기업:
{state['company']}

실제 뉴스/공시 Search Tool은
현재 연결하지 않았습니다.

따라서 최신 사실을 만들지 말고,

- 찾아야 할 뉴스
- 공시
- Catalyst
- Risk
- 출처 관리 방법

을 작성하세요.
"""
    )

    return {
        "research":
            response.content
    }


# ==========================================
# 7. Writer LLM
# ==========================================

def writer(state):

    print(
        "▶ Writer LLM / Revision",
        state["revision_count"]
    )

    response = llm.invoke(
        f"""
당신은 Senior Equity Research Analyst입니다.

기업:
{state['company']}

사용자 요청:
{state['request']}

아래 세 Analyst의 결과를 통합하세요.


[Market Agent]

{state['market']}


[Fundamental Agent]

{state['fundamental']}


[Research Agent]

{state['research']}


이전 Critic Feedback:

{state['critic_feedback']}


다음 목차로 작성하세요.

1. Executive Summary
2. Market Analysis
3. Fundamental Analysis
4. News & Catalyst
5. Risk
6. Investment Thesis
7. 추가로 필요한 데이터


규칙:

- 제공되지 않은 숫자를 만들지 않는다.
- 실제 최신 뉴스를 만들지 않는다.
- 데이터 부족을 명확히 밝힌다.
- 과도한 투자 단정을 피한다.
"""
    )

    return {
        "draft_report":
            response.content
    }


# ==========================================
# 8. Critic LLM
# ==========================================

def critic(state):

    print("▶ Critic LLM")

    result = critic_llm.invoke(
        f"""
당신은 금융 AI 보고서의 Critic입니다.

다음 보고서를 검증하세요.

{state['draft_report']}


다음을 검사하세요.

1. 존재하지 않는 숫자를 생성했는가?
2. 실제 최신 데이터를 아는 척했는가?
3. 근거 없이 투자 의견을 단정했는가?
4. Risk가 누락됐는가?
5. 데이터 부족을 명시했는가?
6. Market/Fundamental/Research 결과가
   일관되게 통합됐는가?


문제가 크지 않으면
passed=true.

수정이 필요하면
passed=false와
구체적인 feedback을 반환하세요.
"""
    )

    print(
        "Critic PASS:",
        result.passed
    )

    print(
        "Feedback:",
        result.feedback
    )

    return {
        "critic_passed":
            result.passed,

        "critic_feedback":
            result.feedback
    }


# ==========================================
# 9. Revision
# ==========================================

def revise(state):

    print("▶ Revision")

    return {
        "revision_count":
            state["revision_count"] + 1
    }


# ==========================================
# 10. Critic Routing
# ==========================================

def after_critic(state):

    if state["critic_passed"]:

        return "approval"

    if state["revision_count"] >= 2:

        return "approval"

    return "revise"


# ==========================================
# 11. Human Approval
# ==========================================

def approval(state):

    print("▶ Human Approval")

    decision = interrupt({

        "company":
            state["company"],

        "report":
            state["draft_report"],

        "critic_feedback":
            state["critic_feedback"],

        "question":
            "최종 보고서를 승인하시겠습니까?"
    })

    return {
        "approved":
            bool(decision)
    }


# ==========================================
# 12. Finalize
# ==========================================

def finalize(state):

    print("▶ Finalize")

    if state["approved"]:

        report = (
            state["draft_report"]
            + "\n\n[Human Approved]"
        )

    else:

        report = (
            state["draft_report"]
            + "\n\n[Human Rejected]"
        )

    return {
        "final_report":
            report
    }


# ==========================================
# 13. Graph
# ==========================================

builder = StateGraph(
    EquityResearchState
)


builder.add_node(
    "market",
    market_agent
)

builder.add_node(
    "fundamental",
    fundamental_agent
)

builder.add_node(
    "research",
    research_agent
)

builder.add_node(
    "writer",
    writer
)

builder.add_node(
    "critic",
    critic
)

builder.add_node(
    "revise",
    revise
)

builder.add_node(
    "approval",
    approval
)

builder.add_node(
    "finalize",
    finalize
)


# ==========================================
# 14. 병렬 Analyst 실행
# ==========================================

builder.add_edge(
    START,
    "market"
)

builder.add_edge(
    START,
    "fundamental"
)

builder.add_edge(
    START,
    "research"
)


# ==========================================
# 15. 합류
# ==========================================

builder.add_edge(
    [
        "market",
        "fundamental",
        "research"
    ],
    "writer"
)


builder.add_edge(
    "writer",
    "critic"
)


# ==========================================
# 16. Critic / Revision Loop
# ==========================================

builder.add_conditional_edges(
    "critic",
    after_critic,
    {
        "approval":
            "approval",

        "revise":
            "revise"
    }
)


builder.add_edge(
    "revise",
    "writer"
)


builder.add_edge(
    "approval",
    "finalize"
)

builder.add_edge(
    "finalize",
    END
)


# ==========================================
# 17. Persistence
# ==========================================

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


# ==========================================
# 18. Thread
# ==========================================

config = {
    "configurable": {
        "thread_id":
            "equity-research-001"
    }
}


# ==========================================
# 19. 입력
# ==========================================

input_state = {

    "company":
        "삼성전자",

    "request":
        "투자 관점에서 기업을 분석해줘",

    "market":
        "",

    "fundamental":
        "",

    "research":
        "",

    "draft_report":
        "",

    "critic_passed":
        False,

    "critic_feedback":
        "",

    "revision_count":
        0,

    "approved":
        False,

    "final_report":
        ""
}


# ==========================================
# 20. 실행
# Human Approval에서 정지
# ==========================================

result = graph.invoke(
    input_state,
    config
)


print(
    "\n=== 현재 실행 상태 ==="
)

print(result)

In [ ]:
# 정상적으로 실행되면:

# Market Agent
# Fundamental Agent
# Research Agent
#         ↓
# Writer LLM
#         ↓
# Critic LLM
#         ↓
# PASS / Revision
#         ↓
# Human Approval

In [ ]:
# 에서 멈춥니다.

# 보고서를 확인한 후 다음 셀에서 승인합니다.

In [ ]:
result = graph.invoke(
    Command(
        resume=True
    ),
    config
)


print(
    "\n==========================="
)

print(
    "FINAL EQUITY RESEARCH REPORT"
)

print(
    "==========================="
)

print(
    result["final_report"]
)

- 최종 구조는 이제 이렇게 됩니다.

                         USER
                           │
                           ▼
                       LangGraph
                           │
          ┌────────────────┼────────────────┐
          ▼                ▼                ▼
    Market LLM      Fundamental LLM    Research LLM
          │                │                │
          └────────────────┼────────────────┘
                           ▼
                      Writer LLM
                           │
                           ▼
                      Critic LLM
                           │
                 ┌─────────┴─────────┐
                 │                   │
               PASS                FAIL
                 │                   │
                 │              Revision
                 │                   │
                 │              Writer LLM
                 │                   │
                 │              Critic LLM
                 ▼
             Human Review
                 │
                 ▼
            Final Report